# Website Screenshot Dataset Builder — vibe-coded site detector

Reads `dataset_phishing.csv` directly from the Colab runtime's local
filesystem (no upload step, no Google Drive), keeps only rows where
`status == "legitimate"`, and screenshots every one of them — no `LIMIT`.

**What this notebook does:**
- Reads the CSV from `/content/dataset_phishing.csv`.
- Keeps only rows where `status` (configurable) equals `"legitimate"`.
- Visits every `url` with headless Chromium, waits for the page to load.
- Takes several screenshots per site at different scroll depths, so long
  landing pages are represented beyond the hero section.
- Randomly mixes desktop and real mobile-device viewports per site (~30–35%
  mobile overall, mix/order differs per site). Mobile uses genuine device
  emulation (viewport, UA, touch, DPR), not a crop of the desktop layout.
- Skips sites that fail to load / time out and keeps going.
- Writes a flat folder of PNGs (`dataset/`) plus `manifest.csv`, both on the
  runtime's local disk.
- Zipping + downloading the images folder is a separate final cell.

**Note:** the local runtime disk is wiped if the session disconnects or
times out. With thousands of `legitimate` rows this run can take hours —
keep the tab open/active, and if it does disconnect partway you'll need to
re-upload the CSV and start over (there's nothing persisted outside the
runtime here, by design, since Drive is off the table).

## 1. Setup

In [ ]:
!pip install -q playwright nest_asyncio
!playwright install --with-deps chromium

In [ ]:
import asyncio
import nest_asyncio
nest_asyncio.apply()  # lets asyncio play nicely inside Colab's existing event loop

## 2. Check the CSV is there

`dataset_phishing.csv` should already be sitting in the Colab runtime's file browser (`/content/`). This just confirms it before we start.

In [ ]:
import os

CSV_PATH = "/content/dataset_phishing.csv"
assert os.path.exists(CSV_PATH), f"{CSV_PATH} not found — check the Colab file browser on the left."
print("Found:", CSV_PATH, f"({os.path.getsize(CSV_PATH)} bytes)")

## 3. Core pipeline code

Just defines functions — nothing runs yet.

In [ ]:
import csv
import random
import re
import time
from pathlib import Path
from urllib.parse import urlparse

from playwright.async_api import async_playwright, TimeoutError as PWTimeoutError

DESKTOP_VIEWPORTS = [
    {"width": 1920, "height": 1080},
    {"width": 1536, "height": 864},
    {"width": 1440, "height": 900},
    {"width": 1366, "height": 768},
    {"width": 1280, "height": 800},
]

# Real device profiles -> genuine responsive/mobile rendering, not a crop.
MOBILE_DEVICES = [
    "iPhone 13",
    "iPhone 12 Pro",
    "iPhone SE",
    "Pixel 5",
    "Galaxy S9+",
]

FILENAME_RE = re.compile(r"^(?P<slug>.+)_(?P<mode>desktop|mobile)_(?P<idx>\d+)\.png$")
COUNTERS = {}
MANIFEST_ROWS = []


def sanitize(name: str) -> str:
    name = (name or "").strip().lower()
    name = re.sub(r"[^a-z0-9_-]+", "-", name)
    name = re.sub(r"-{2,}", "-", name).strip("-_")
    return (name or "site")[:80]


def derive_slug_from_url(url: str) -> str:
    p = urlparse(url)
    host = (p.netloc or p.path).split(":")[0]
    if host.startswith("www."):
        host = host[4:]
    slug = host
    path = p.path.strip("/")
    if path:
        slug += "-" + path.replace("/", "-")
    return sanitize(slug)


def load_legitimate_sites(csv_path, status_field="status", status_value="legitimate", url_field="url"):
    total = 0
    kept = []
    with open(csv_path, newline="", encoding="utf-8-sig") as f:
        reader = csv.DictReader(f)
        for row in reader:
            total += 1
            status = (row.get(status_field) or "").strip().lower()
            if status != status_value.strip().lower():
                continue
            url = (row.get(url_field) or "").strip()
            if not url:
                continue
            site = dict(row)
            site["live_url"] = url
            if not site.get("slug"):
                site["slug"] = derive_slug_from_url(url)
            kept.append(site)
    print(f"Loaded {total} row(s) from {csv_path}; "
          f"{len(kept)} have {status_field}=\'{status_value}\' and a usable {url_field}.")
    return kept


def init_counters(out_dir: Path):
    COUNTERS.clear()
    for f in out_dir.glob("*.png"):
        m = FILENAME_RE.match(f.name)
        if not m:
            continue
        key = (m.group("slug"), m.group("mode"))
        COUNTERS[key] = max(COUNTERS.get(key, 0), int(m.group("idx")))


def slugs_with_existing(out_dir: Path):
    existing = set()
    for f in out_dir.glob("*.png"):
        m = FILENAME_RE.match(f.name)
        if m:
            existing.add(m.group("slug"))
    return existing


def next_index(slug, mode):
    key = (slug, mode)
    COUNTERS[key] = COUNTERS.get(key, 0) + 1
    return COUNTERS[key]


def pick_scroll_positions(max_scroll, viewport_h, count):
    if max_scroll < max(40, viewport_h * 0.1):
        return [0]
    fractions = [0.0] if count <= 1 else [i / (count - 1) for i in range(count)]
    positions = []
    for frac in fractions:
        jitter = random.uniform(-0.04, 0.04)
        f = min(1.0, max(0.0, frac + jitter))
        positions.append(int(f * max_scroll))
    positions = sorted(set(positions))
    min_gap = max(40, int(viewport_h * 0.15))
    deduped = []
    for y in positions:
        if not deduped or y - deduped[-1] >= min_gap:
            deduped.append(y)
    return deduped


async def capture_viewport(browser, pw, site, slug, url, mode, count, timeout_ms, out_dir: Path):
    if mode == "desktop":
        vp = random.choice(DESKTOP_VIEWPORTS)
        context = await browser.new_context(viewport=vp)
        viewport_w, viewport_h = vp["width"], vp["height"]
    else:
        device = pw.devices[random.choice(MOBILE_DEVICES)]
        context = await browser.new_context(**device)
        viewport_w = device["viewport"]["width"]
        viewport_h = device["viewport"]["height"]

    page = await context.new_page()
    saved = 0
    try:
        try:
            await page.goto(url, wait_until="load", timeout=timeout_ms)
        except PWTimeoutError:
            print(f"[warn] {slug} ({mode}): load timed out, continuing anyway")

        try:
            await page.wait_for_load_state("networkidle", timeout=8000)
        except PWTimeoutError:
            pass
        await page.wait_for_timeout(400)

        try:
            scroll_height = await page.evaluate(
                "Math.max(document.body.scrollHeight, document.documentElement.scrollHeight)"
            )
        except Exception:
            scroll_height = viewport_h

        max_scroll = max(0, scroll_height - viewport_h)
        positions = pick_scroll_positions(max_scroll, viewport_h, count)

        for y in positions[:count]:
            try:
                await page.evaluate(f"window.scrollTo(0, {y})")
            except Exception:
                pass
            await page.wait_for_timeout(random.randint(350, 700))

            idx = next_index(slug, mode)
            fname = f"{slug}_{mode}_{idx:03d}.png"
            fpath = out_dir / fname
            try:
                await page.screenshot(path=str(fpath))
                saved += 1
                row = {
                    "filename": fname,
                    "slug": slug,
                    "mode": mode,
                    "index": idx,
                    "scroll_y": y,
                    "viewport_width": viewport_w,
                    "viewport_height": viewport_h,
                }
                for k, v in site.items():
                    if k not in row:
                        row[k] = v
                MANIFEST_ROWS.append(row)
            except Exception as e:
                print(f"[warn] {slug} ({mode}): screenshot failed at y={y}: {e}")

    except Exception as e:
        print(f"[error] {slug} ({mode}): {e}")
    finally:
        await context.close()

    return saved


async def process_site(site, sem, pw, out_dir, min_shots, max_shots, mobile_ratio,
                        timeout_ms, headless, existing_slugs):
    slug = sanitize(site.get("slug") or "")
    url = site.get("live_url")

    if not url:
        return

    if slug in existing_slugs:
        return

    async with sem:
        n_total = random.randint(min_shots, max_shots)
        ratio = min(1.0, max(0.0, mobile_ratio + random.uniform(-0.08, 0.08)))
        mobile_count = min(n_total, round(n_total * ratio))
        if n_total >= 2:
            mobile_count = min(max(mobile_count, 1), n_total - 1) if random.random() < 0.9 else mobile_count
        desktop_count = n_total - mobile_count

        browser = None
        try:
            browser = await pw.chromium.launch(headless=headless)
            saved = 0
            if desktop_count > 0:
                saved += await capture_viewport(browser, pw, site, slug, url, "desktop",
                                                 desktop_count, timeout_ms, out_dir)
            if mobile_count > 0:
                saved += await capture_viewport(browser, pw, site, slug, url, "mobile",
                                                 mobile_count, timeout_ms, out_dir)

            if saved == 0:
                print(f"[fail] {slug}: unreachable or produced no screenshots ({url})")
            else:
                print(f"[ok]   {slug}: saved {saved} screenshot(s) "
                      f"(target {desktop_count} desktop / {mobile_count} mobile)")
        except Exception as e:
            print(f"[error] {slug}: {e} ({url})")
        finally:
            if browser is not None:
                await browser.close()


async def run_pipeline(input_csv, output_dir, manifest_path, status_field="status",
                        status_value="legitimate", url_field="url", min_shots=4, max_shots=7,
                        mobile_ratio=0.32, concurrency=3, timeout_ms=30000, headless=True,
                        skip_existing=True):
    sites = load_legitimate_sites(input_csv, status_field, status_value, url_field)

    out_dir = Path(output_dir)
    out_dir.mkdir(parents=True, exist_ok=True)
    init_counters(out_dir)
    existing_slugs = slugs_with_existing(out_dir) if skip_existing else set()
    if existing_slugs:
        print(f"{len(existing_slugs)} site(s) already have screenshots and will be skipped.")

    sem = asyncio.Semaphore(concurrency)
    start = time.time()
    async with async_playwright() as pw:
        tasks = [
            process_site(site, sem, pw, out_dir, min_shots, max_shots, mobile_ratio,
                          timeout_ms, headless, existing_slugs)
            for site in sites
        ]
        await asyncio.gather(*tasks)

    elapsed = time.time() - start
    total_images = len(list(out_dir.glob("*.png")))
    print(f"\nDone in {elapsed:.1f}s. {total_images} PNG screenshot(s) total in {out_dir}/")

    all_rows = []
    manifest_file = Path(manifest_path)
    if manifest_file.exists():
        with open(manifest_file, newline="", encoding="utf-8") as f:
            all_rows = list(csv.DictReader(f))
    all_rows.extend(MANIFEST_ROWS)

    if all_rows:
        fieldnames = list(all_rows[0].keys())
        for r in all_rows[1:]:
            for k in r.keys():
                if k not in fieldnames:
                    fieldnames.append(k)
        with open(manifest_file, "w", newline="", encoding="utf-8") as f:
            writer = csv.DictWriter(f, fieldnames=fieldnames, restval="")
            writer.writeheader()
            writer.writerows(all_rows)
        print(f"{manifest_path} now has {len(all_rows)} row(s) ({len(MANIFEST_ROWS)} new this run).")
    else:
        print("No new screenshots captured this run.")

    MANIFEST_ROWS.clear()

## 4. Config

- No `LIMIT`: every `legitimate` row in the CSV gets processed. With
  thousands of rows this will run for hours; that's expected.
- Everything stays local to this Colab runtime — `OUTPUT_DIR` and
  `MANIFEST_PATH` are plain relative paths under `/content/`.

In [ ]:
OUTPUT_DIR = "dataset"
MANIFEST_PATH = "manifest.csv"

STATUS_FIELD = "status"       # column name holding the status
STATUS_VALUE = "legitimate"   # only rows with this value are processed
URL_FIELD = "url"             # column name holding the website URL

MIN_SHOTS = 4          # screenshots per site (min)
MAX_SHOTS = 7          # screenshots per site (max)
MOBILE_RATIO = 0.32    # target ~30-35% of shots as mobile
CONCURRENCY = 3        # sites processed in parallel — keep modest on Colab
TIMEOUT_MS = 30000     # page load timeout per site
HEADLESS = True
SKIP_EXISTING = True   # don't reprocess a slug that already has screenshots (useful if the cell is re-run without restarting the runtime)

## 5. Run

This processes every `legitimate` row — expect a long run. If the runtime disconnects or restarts, the local disk is wiped and this will need to start over from `dataset_phishing.csv` again.

In [ ]:
await run_pipeline(
    CSV_PATH, OUTPUT_DIR, MANIFEST_PATH,
    status_field=STATUS_FIELD, status_value=STATUS_VALUE, url_field=URL_FIELD,
    min_shots=MIN_SHOTS, max_shots=MAX_SHOTS, mobile_ratio=MOBILE_RATIO,
    concurrency=CONCURRENCY, timeout_ms=TIMEOUT_MS, headless=HEADLESS,
    skip_existing=SKIP_EXISTING,
)

## 6. Quick peek (optional)

Show a couple of captured images inline and the first few manifest rows, as a sanity check.

In [ ]:
import pandas as pd
from IPython.display import Image, display

df = pd.read_csv(MANIFEST_PATH)
print(df.shape)
display(df.head())

for fname in df["filename"].head(2):
    display(Image(filename=str(Path(OUTPUT_DIR) / fname), width=400))

## 7. Zip and download the images

Separate step — zips the flat images folder and downloads it, plus `manifest.csv`.

In [ ]:
import shutil
from google.colab import files as gfiles

zip_name = "vibecoded_screenshots"
shutil.make_archive(zip_name, "zip", OUTPUT_DIR)
print(f"Zipped {OUTPUT_DIR}/ -> {zip_name}.zip")

gfiles.download(f"{zip_name}.zip")
gfiles.download(MANIFEST_PATH)